# FunSearch DEC Cloud API 实验（Colab）

本 notebook 目标：在 Colab 上一键跑通云端 API 版 FunSearch + Two-Stage DEC，并输出可用于对比的指标文件。

## 你关心的 4 个点
- 与本地 LLM 版 **同一套去重实现**（Stage1 + Stage2，代码路径一致）。
- 默认已使用当前推荐参数：`stage1=15`、`stage2=256`、`max_non_code_retries=2`。
- 支持**多数据集模式**：一条启发式在所有数据集上评分，更真实地评估泛化能力。
- 包含结果统计展示与指标解释（Time Saved / API Efficiency / False Positive Rate / Quality）。


## 0) 运行前准备
1. 在 Colab `Runtime` 中启用 Python 3。
2. 准备好仓库地址和云端 API Key。
3. 如需代理，填写 `HTTPS_PROXY/HTTP_PROXY`。


In [ ]:
# 1) 克隆仓库（修改为你的仓库）
REPO_URL = "https://github.com/ultrabababa/FunSearch-DEC.git"
!git clone $REPO_URL
%cd FunSearch-DEC

In [ ]:
# 2) 安装依赖
!python -m pip install -U pip
!python -m pip install -r requirements.txt
!python -m pip install pandas matplotlib

In [ ]:
# 3) 云端 API 配置（必填）
import os

os.environ['FUNSEARCH_CLOUD_API_KEY'] = '<YOUR_API_KEY>'
os.environ['FUNSEARCH_CLOUD_BASE_URL'] = 'https://api.bltcy.ai'
os.environ['FUNSEARCH_CLOUD_MODEL'] = 'gpt-5-nano'

# 固定要求：关闭思考模式，仅输出代码
os.environ['FUNSEARCH_DISABLE_THINKING'] = 'on'
os.environ['FUNSEARCH_THINKING_PARAM_MODE'] = 'both'

# 启用 OpenAI SDK 路径（云端 API 必须）
os.environ['FUNSEARCH_USE_OPENAI_SDK'] = '1'
# 关闭 reasoning 模型的 thinking 过程以提高速度
os.environ['FUNSEARCH_REASONING_EFFORT'] = 'none'
os.environ['FUNSEARCH_MAX_NON_CODE_RETRIES'] = '8'
os.environ['FUNSEARCH_VERBOSE_SAMPLES'] = '1'

In [ ]:
# 4) 连通性检查（先确保这个通过）
!python tools/test_cloud_api_config.py --timeout 30

## 5) 快速 Smoke 验证（单数据集，repeats=1）
- 先用这个快速验证完整链路没问题，再跑全量。
- 使用 OR_u1000 数据集（1000 items，评估成本高，能体现 DEC 效果）


In [ ]:
# 点击运行本单元 = 执行快速 smoke (repeats=1)
SMOKE_DATASET = 'OR_u1000'
SMOKE_MAX_SAMPLES = 5
SMOKE_REPEATS = 1

!python tools/run_experiment_matrix.py \
  --dataset "$SMOKE_DATASET" \
  --max-samples $SMOKE_MAX_SAMPLES \
  --repeats $SMOKE_REPEATS \
  --stage1-case-count 15 \
  --stage2-random-cases 256

!python tools/summarize_experiment_matrix.py --dataset "$SMOKE_DATASET" --repeats $SMOKE_REPEATS

## 6) 正式全量实验（多数据集模式）
- 这一步会自动：
  - 跑矩阵 baseline/dedup
  - 每条启发式在**所有数据集**上评分（更真实评估泛化能力）
  - 使用优化后的参数：stage1=15, stage2=256
- 预计时间：约 2-4 小时（取决于 API 响应速度）


In [ ]:
# 多数据集模式：所有 OR-Library 数据集
MULTI_DATASET_KEYS = "OR_u120,OR_u250,OR_u500,OR_u1000,OR_t60,OR_t120,OR_t249,OR_t501"

!python tools/run_experiment_matrix.py \
  --dataset-keys "$MULTI_DATASET_KEYS" \
  --max-samples 20 \
  --repeats 3 \
  --stage1-case-count 15 \
  --stage2-random-cases 256

In [ ]:
# 7) 生成汇总结果
!python tools/summarize_experiment_matrix.py \
  --dataset-keys "$MULTI_DATASET_KEYS" \
  --repeats 3

In [ ]:
# 8) 读取 aggregate 结果
import json
from pathlib import Path

# 找到最新的 summary 文件
import glob
summary_files = sorted(glob.glob('logs/experiments/summary_*.json'))
if summary_files:
    latest = summary_files[-1]
    data = json.loads(Path(latest).read_text(encoding='utf-8'))
    print(f'\n=== {Path(latest).name} ===')
    print(json.dumps(data.get('aggregate', {}), indent=2, ensure_ascii=False))

## 9) 指标解释（与 Proposal 对齐）

| Proposal 指标 | 对应字段 | 说明 |
|--------------|----------|------|
| **Time Saved** | `time_saved_ratio` / `pipeline_time_saved_ratio` | 评估时间节省比例（>0 更好） |
| **API Sample Efficiency** | `dedup_dedup_hits` / `dedup_sandbox_evals` | 去重拦截的样本占比（越高越高效） |
| **False Positive Rate** | `dedup_stage2_collision_reject_rate` | Stage 2 正确拒绝率（>95% 为佳） |
| **Performance Quality** | `best_score_diff_dedup_minus_baseline` | 分数差（>0 代表 dedup 分数更好） |

### 关键结果解读
- 如果 `time_saved_ratio > 0`：DEC 成功节省了评估时间
- 如果 `best_score_diff > 0`：DEC 版本找到了更好的启发式
- 如果 `stage2_collision_reject_rate > 0.95`：Stage 2 过滤有效，误判率低

In [ ]:
# 10) 表格与图（快速可视化）
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import glob

# 找到最新的 summary CSV
csv_files = sorted(glob.glob('logs/experiments/summary_*.csv'))
if csv_files:
    latest_csv = csv_files[-1]
    df = pd.read_csv(latest_csv)
    
    print(f'=== {Path(latest_csv).name} ===')
    display(df[['repeat', 'dataset', 'baseline_best_score', 'dedup_best_score', 
                'best_score_diff_dedup_minus_baseline', 'time_saved_ratio', 
                'pipeline_time_saved_ratio', 'dedup_dedup_hits']].to_string())
    
    # 绘图
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Time Saved
    axes[0,0].bar(df['repeat'], df['time_saved_ratio'])
    axes[0,0].axhline(0, color='gray', linestyle='--')
    axes[0,0].set_title('Time Saved Ratio')
    axes[0,0].set_xlabel('Repeat')
    axes[0,0].set_ylabel('Ratio')
    
    # Pipeline Time Saved
    axes[0,1].bar(df['repeat'], df['pipeline_time_saved_ratio'], color='tab:orange')
    axes[0,1].axhline(0, color='gray', linestyle='--')
    axes[0,1].set_title('Pipeline Time Saved Ratio')
    axes[0,1].set_xlabel('Repeat')
    axes[0,1].set_ylabel('Ratio')
    
    # Score Comparison
    x = range(len(df))
    width = 0.35
    axes[1,0].bar([i - width/2 for i in x], df['baseline_best_score'], width, label='Baseline')
    axes[1,0].bar([i + width/2 for i in x], df['dedup_best_score'], width, label='Dedup')
    axes[1,0].set_title('Best Score Comparison')
    axes[1,0].set_xlabel('Repeat')
    axes[1,0].set_ylabel('Score')
    axes[1,0].legend()
    
    # Dedup Hits
    axes[1,1].bar(df['repeat'], df['dedup_dedup_hits'], color='tab:green')
    axes[1,1].set_title('Dedup Hits')
    axes[1,1].set_xlabel('Repeat')
    axes[1,1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 11) 下载关键结果文件
from google.colab import files
import glob

# 下载所有 summary 文件
for f in sorted(glob.glob('logs/experiments/summary_*')):
    print(f'Downloading: {f}')
    files.download(f)